# DEAI-opdrachten Classificatie – Ames Housing
**Binaire classificatie:** voorspellen of een huis wel/geen garage heeft
**Multi-class classificatie:** voorspellen van het kwaliteitsniveau (Overall Qual)

In [1]:
import warnings
warnings.filterwarnings("ignore")

import logging
import pandas as pd
import numpy as np

from IPython.display import display
from pandas.api.types import is_numeric_dtype

from sklearn.model_selection import train_test_split
# Splitst de data in trainingsdata en testdata

from sklearn.tree import DecisionTreeClassifier
# Classificatiemodel dat voorspelt bij welke klasse een voorbeeld hoort

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
# Metrics om een classificatiemodel te beoordelen

In [2]:
logger = logging.getLogger("classification_notebook")
logger.setLevel(logging.INFO)
# Maakt een logger aan voor informatieve meldingen

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
    logger.addHandler(handler)
    # Zorgt dat logberichten netjes in de output verschijnen

logger.propagate = False
# Voorkomt dubbele logberichten

logger.info("Classificatie-notebook is gestart")

INFO - Classificatie-notebook is gestart


In [3]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
# Zorgt dat meer kolommen en bredere tabellen zichtbaar zijn in de output

In [4]:
bestand = "AmesHousing.xlsx"

df = pd.read_excel(bestand, sheet_name="AmesHousing")
data_dictionary = pd.read_excel(bestand, sheet_name="Data Dictionary")
# Leest de dataset en de data dictionary uit het Excel-bestand in

logger.info(f"Dataset ingeladen met shape: {df.shape}")
# Laat zien hoeveel rijen en kolommen de dataset heeft

display(df.head())
# Toont de eerste rijen van de dataset als controle

display(data_dictionary.head(20))
# Toont een deel van de data dictionary om de kolommen te begrijpen

INFO - Dataset ingeladen met shape: (2930, 12)


,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story


,Variabele,Betekenis
0,ID,"Uniek nummer per huis, te vergelijken met een ..."
1,SalePrice,Verkoopprijs van het huis (in dollars: USD)
2,Garage,Geeft weer of het huis wel/geen garage bevat
3,Overall Qual,Algemene kwaliteit van materialen en afwerking...
4,Gr Liv Area,Woonoppervlak boven de grond (square feet)
5,Total Bsmt SF,Totale oppervlakte van de kelder
6,Lot Area,Grootte van het perceel (square feet)
7,Year Built,Bouwjaar van het huis
8,Full Bath,Aantal volledige badkamers
9,Bedroom AbvGr,Aantal slaapkamers boven de grond


In [5]:
# help(DecisionTreeClassifier)
# Kan gebruikt worden om alle beschikbare hyperparameters van het model te bekijken

logger.info("Gebruikte hyperparameters: max_depth, min_samples_split, min_samples_leaf")

INFO - Gebruikte hyperparameters: max_depth, min_samples_split, min_samples_leaf


In [6]:
# Maakt de invoerdata klaar voor het classificatiemodel
def maak_features(dataframe, feature_kolommen):
    X = dataframe[feature_kolommen].copy()
    # Neemt alleen de gekozen featurekolommen over

    for kolom in X.columns:
        if is_numeric_dtype(X[kolom]):
            X[kolom] = X[kolom].fillna(X[kolom].median())
        else:
            modus = X[kolom].mode(dropna=True)
            if len(modus) > 0:
                X[kolom] = X[kolom].fillna(modus.iloc[0])
            else:
                X[kolom] = X[kolom].fillna("Onbekend")
    # Vult missende waarden op: mediaan bij numeriek, modus of 'Onbekend' bij categorisch

    categorische_kolommen = X.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()
    # Zoekt welke kolommen categorisch zijn

    X = pd.get_dummies(X, columns=categorische_kolommen, drop_first=False)
    # Zet categorische kolommen om naar dummyvariabelen

    return X
    # Geeft de verwerkte featuredata terug

In [7]:
def run_experiment(dataframe, target_kolom, feature_kolommen, hyperparameters):
    # Voert één classificatie-experiment uit en geeft de resultaten terug.

    data = dataframe[feature_kolommen + [target_kolom]].copy()
    data = data.dropna(subset=[target_kolom])
    # Maakt een dataset met de gekozen features en target, en verwijdert rijen zonder target

    X = maak_features(data, feature_kolommen)
    y = data[target_kolom].copy()
    # Maakt de featuredata en doelvariabele klaar

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    # Splitst de data in trainingsdata en testdata, met behoud van de klassedistributie

    logger.info(f"Train-test-split uitgevoerd: X_train={X_train.shape}, X_test={X_test.shape}")

    model = DecisionTreeClassifier(random_state=42, **hyperparameters)
    # Maakt een Decision Tree Classifier met de opgegeven hyperparameters

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    # Traint het model en maakt voorspellingen op de testdata

    labels = sorted(y.unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    # Maakt de confusion matrix op basis van alle klassen

    resultaat = {
        "features": feature_kolommen,
        "hyperparameters": hyperparameters,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "confusion_matrix": cm,
        "labels": labels,
        "report": classification_report(y_test, y_pred, zero_division=0),
        "X_train_shape": X_train.shape,
        "X_test_shape": X_test.shape,
        "y_train_shape": y_train.shape,
        "y_test_shape": y_test.shape
    }
    # Slaat de instellingen en prestaties van het experiment op

    logger.info(
        f"Experiment afgerond | accuracy={resultaat['accuracy']:.3f}, "
        f"f1_macro={resultaat['f1_macro']:.3f}"
    )

    return resultaat

In [8]:
def maak_resultaten_tabel(resultaten_dict):
    rijen = []
    # Lege lijst waarin de resultaten per run worden verzameld

    for naam, res in resultaten_dict.items():
        rijen.append({
            "Run": naam,
            "Features": ", ".join(res["features"]),
            "Hyperparameters": str(res["hyperparameters"]),
            "Accuracy": round(res["accuracy"], 4),
            "Precision macro": round(res["precision_macro"], 4),
            "Recall macro": round(res["recall_macro"], 4),
            "F1 macro": round(res["f1_macro"], 4)
        })
        # Zet per experiment de belangrijkste resultaten in een rij

    resultaten_tabel = pd.DataFrame(rijen)
    # Maakt van alle rijen samen een overzichtelijke tabel

    logger.info(f"Resultatentabel aangemaakt met {len(resultaten_tabel)} runs")
    return resultaten_tabel

In [9]:
garage_target = "Garage"
garage_features_eerste_run = ["SalePrice", "Gr Liv Area", "Neighborhood"]
# Doelvariabele en gekozen features voor het eerste garage-experiment

garage_data = df[garage_features_eerste_run + [garage_target]].dropna(subset=[garage_target]).copy()
# Maakt een dataset met de gekozen features en target, en verwijdert rijen zonder target

X_garage = maak_features(garage_data, garage_features_eerste_run)
y_garage = garage_data[garage_target].copy()
# Bereidt de feature data voor en maakt de doelvariabele apart

logger.info(f"Featurematrix aangemaakt: X_garage={X_garage.shape}, y_garage={y_garage.shape}")
# Laat zien hoe groot de invoerdata en target zijn

X_train_garage, X_test_garage, y_train_garage, y_test_garage = train_test_split(
    X_garage, y_garage, test_size=0.20, random_state=42, stratify=y_garage
)
# Splitst de data in trainingsdata en testdata, met behoud van de klassedistributie

logger.info(
    f"Train-test-split uitgevoerd: "
    f"X_train={X_train_garage.shape}, X_test={X_test_garage.shape}, "
    f"y_train={y_train_garage.shape}, y_test={y_test_garage.shape}"
)

INFO - Featurematrix aangemaakt: X_garage=(2930, 30), y_garage=(2930,)
INFO - Train-test-split uitgevoerd: X_train=(2344, 30), X_test=(586, 30), y_train=(2344,), y_test=(586,)


In [10]:
garage_experimenten = {
    "Initial run": {
        "features": ["SalePrice", "Gr Liv Area", "Neighborhood"],
        "params": {"max_depth": 4, "min_samples_split": 10},
        "reden": "Starten met de verwachte top 3 features."
    },
    "Experiment 1": {
        "features": ["SalePrice", "Gr Liv Area", "Neighborhood"],
        "params": {"max_depth": 6, "min_samples_split": 10},
        "reden": "Kijken of een diepere boom betere patronen vindt."
    },
    "Experiment 2": {
        "features": ["SalePrice", "Gr Liv Area", "Year Built", "Neighborhood"],
        "params": {"max_depth": 4, "min_samples_split": 10},
        "reden": "Extra feature Year Built toevoegen."
    },
    "Experiment 3": {
        "features": ["SalePrice", "Gr Liv Area", "Year Built", "Neighborhood"],
        "params": {"max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 2},
        "reden": "Nog een keer tunen met extra feature en extra hyperparameter."
    }
}
# Bevat de verschillende classificatie-experimenten met features, hyperparameters en reden

garage_resultaten = {}
# Lege dictionary om de resultaten van alle experimenten op te slaan

for naam, info in garage_experimenten.items():
    logger.info(f"Start experiment: {naam} | reden: {info['reden']}")

    garage_resultaten[naam] = run_experiment(
        df,
        target_kolom="Garage",
        feature_kolommen=info["features"],
        hyperparameters=info["params"]
    )
    # Voert elk experiment uit en slaat het resultaat op onder de naam van de run

garage_tabel = maak_resultaten_tabel(garage_resultaten)
# Zet alle resultaten om in een overzichtelijke tabel

display(garage_tabel)
# Toont de resultaten van alle experimenten

INFO - Start experiment: Initial run | reden: Starten met de verwachte top 3 features.
INFO - Train-test-split uitgevoerd: X_train=(2344, 30), X_test=(586, 30)
INFO - Experiment afgerond | accuracy=0.945, f1_macro=0.515
INFO - Start experiment: Experiment 1 | reden: Kijken of een diepere boom betere patronen vindt.
INFO - Train-test-split uitgevoerd: X_train=(2344, 30), X_test=(586, 30)
INFO - Experiment afgerond | accuracy=0.942, f1_macro=0.538
INFO - Start experiment: Experiment 2 | reden: Extra feature Year Built toevoegen.
INFO - Train-test-split uitgevoerd: X_train=(2344, 31), X_test=(586, 31)
INFO - Experiment afgerond | accuracy=0.947, f1_macro=0.567
INFO - Start experiment: Experiment 3 | reden: Nog een keer tunen met extra feature en extra hyperparameter.
INFO - Train-test-split uitgevoerd: X_train=(2344, 31), X_test=(586, 31)
INFO - Experiment afgerond | accuracy=0.945, f1_macro=0.689
INFO - Resultatentabel aangemaakt met 4 runs


,Run,Features,Hyperparameters,Accuracy,Precision macro,Recall macro,F1 macro
0,Initial run,"SalePrice, Gr Liv Area, Neighborhood","{'max_depth': 4, 'min_samples_split': 10}",0.9454,0.7235,0.5147,0.5154
1,Experiment 1,"SalePrice, Gr Liv Area, Neighborhood","{'max_depth': 6, 'min_samples_split': 10}",0.9420,0.6408,0.5276,0.5376
2,Experiment 2,"SalePrice, Gr Liv Area, Year Built, Neighborhood","{'max_depth': 4, 'min_samples_split': 10}",0.9471,0.7750,0.5451,0.5674
3,Experiment 3,"SalePrice, Gr Liv Area, Year Built, Neighborhood","{'max_depth': 10, 'min_samples_split': 10, 'mi...",0.9454,0.7314,0.6619,0.6894


In [11]:
beste_garage_run = garage_tabel.sort_values("F1 macro", ascending=False).iloc[0]["Run"]
# Zoekt de run met de hoogste F1 macro-score

beste_garage = garage_resultaten[beste_garage_run]
# Haalt de volledige resultaten van de beste garage-run op

logger.info(f"Beste garage-run op basis van F1 macro: {beste_garage_run}")
# Laat zien welke run het best presteerde

logger.info("Classification report van de beste garage-run:")
print(beste_garage["report"])
# Toont het classificatierapport met precision, recall en F1-score per klasse

garage_cm_df = pd.DataFrame(
    beste_garage["confusion_matrix"],
    index=[f"Werkelijk: {label}" for label in beste_garage["labels"]],
    columns=[f"Voorspeld: {label}" for label in beste_garage["labels"]]
)
# Zet de confusion matrix om in een overzichtelijke tabel

display(garage_cm_df)
# Toont de confusion matrix van de beste run

INFO - Beste garage-run op basis van F1 macro: Experiment 3
INFO - Classification report van de beste garage-run:


              precision    recall  f1-score   support

          no       0.50      0.34      0.41        32
         yes       0.96      0.98      0.97       554

    accuracy                           0.95       586
   macro avg       0.73      0.66      0.69       586
weighted avg       0.94      0.95      0.94       586



,Voorspeld: no,Voorspeld: yes
Werkelijk: no,11,21
Werkelijk: yes,11,543


In [12]:
qual_target = "Overall Qual"
qual_features_eerste_run = ["SalePrice", "Year Built", "Neighborhood"]
# Doelvariabele en gekozen features voor het eerste quality-experiment

qual_data = df[qual_features_eerste_run + [qual_target]].dropna(subset=[qual_target]).copy()
# Maakt een dataset met de gekozen features en target, en verwijdert rijen zonder target

X_qual = maak_features(qual_data, qual_features_eerste_run)
y_qual = qual_data[qual_target].copy()
# Bereidt de featuredata voor en maakt de doelvariabele apart

logger.info(f"Featurematrix aangemaakt: X_qual={X_qual.shape}, y_qual={y_qual.shape}")
# Laat zien hoe groot de featuredata en target zijn

X_train_qual, X_test_qual, y_train_qual, y_test_qual = train_test_split(
    X_qual, y_qual, test_size=0.20, random_state=42, stratify=y_qual
)
# Splitst de data in trainingsdata en testdata, met behoud van de klassedistributie

logger.info(
    f"Train-test-split uitgevoerd: "
    f"X_train={X_train_qual.shape}, X_test={X_test_qual.shape}, "
    f"y_train={y_train_qual.shape}, y_test={y_test_qual.shape}"
)

INFO - Featurematrix aangemaakt: X_qual=(2930, 30), y_qual=(2930,)
INFO - Train-test-split uitgevoerd: X_train=(2344, 30), X_test=(586, 30), y_train=(2344,), y_test=(586,)


In [13]:
qual_experimenten = {
    "Initial run": {
        "features": ["SalePrice", "Year Built", "Neighborhood"],
        "params": {"max_depth": 4, "min_samples_split": 20},
        "reden": "Starten met de verwachte top 3 features."
    },
    "Experiment 1": {
        "features": ["SalePrice", "Year Built", "Neighborhood"],
        "params": {"max_depth": 7, "min_samples_split": 20, "min_samples_leaf": 5},
        "reden": "Eerst kijken of betere hyperparameters al helpen."
    },
    "Experiment 2": {
        "features": ["SalePrice", "Year Built", "Gr Liv Area", "Neighborhood"],
        "params": {"max_depth": 6, "min_samples_split": 10},
        "reden": "Daarna extra feature Gr Liv Area toevoegen."
    },
    "Experiment 3": {
        "features": ["SalePrice", "Year Built", "Gr Liv Area", "Neighborhood", "House Style"],
        "params": {"max_depth": 6, "min_samples_split": 10},
        "reden": "Nog een categorische feature toevoegen."
    }
}
# Bevat de verschillende experimenten voor het voorspellen van Overall Qual

qual_resultaten = {}
# Lege dictionary om de resultaten van alle experimenten op te slaan

for naam, info in qual_experimenten.items():
    logger.info(f"Start experiment: {naam} | reden: {info['reden']}")

    qual_resultaten[naam] = run_experiment(
        df,
        target_kolom="Overall Qual",
        feature_kolommen=info["features"],
        hyperparameters=info["params"]
    )
    # Voert elk experiment uit en slaat het resultaat op

qual_tabel = maak_resultaten_tabel(qual_resultaten)
# Zet alle resultaten om in een overzichtelijke tabel

display(qual_tabel)
# Toont de resultaten van alle experimenten

INFO - Start experiment: Initial run | reden: Starten met de verwachte top 3 features.
INFO - Train-test-split uitgevoerd: X_train=(2344, 30), X_test=(586, 30)
INFO - Experiment afgerond | accuracy=0.532, f1_macro=0.310
INFO - Start experiment: Experiment 1 | reden: Eerst kijken of betere hyperparameters al helpen.
INFO - Train-test-split uitgevoerd: X_train=(2344, 30), X_test=(586, 30)
INFO - Experiment afgerond | accuracy=0.553, f1_macro=0.386
INFO - Start experiment: Experiment 2 | reden: Daarna extra feature Gr Liv Area toevoegen.
INFO - Train-test-split uitgevoerd: X_train=(2344, 31), X_test=(586, 31)
INFO - Experiment afgerond | accuracy=0.548, f1_macro=0.329
INFO - Start experiment: Experiment 3 | reden: Nog een categorische feature toevoegen.
INFO - Train-test-split uitgevoerd: X_train=(2344, 39), X_test=(586, 39)
INFO - Experiment afgerond | accuracy=0.553, f1_macro=0.331
INFO - Resultatentabel aangemaakt met 4 runs


,Run,Features,Hyperparameters,Accuracy,Precision macro,Recall macro,F1 macro
0,Initial run,"SalePrice, Year Built, Neighborhood","{'max_depth': 4, 'min_samples_split': 20}",0.5324,0.3123,0.3190,0.3096
1,Experiment 1,"SalePrice, Year Built, Neighborhood","{'max_depth': 7, 'min_samples_split': 20, 'min...",0.5529,0.4345,0.3730,0.3860
2,Experiment 2,"SalePrice, Year Built, Gr Liv Area, Neighborhood","{'max_depth': 6, 'min_samples_split': 10}",0.5478,0.3294,0.3315,0.3291
3,Experiment 3,"SalePrice, Year Built, Gr Liv Area, Neighborho...","{'max_depth': 6, 'min_samples_split': 10}",0.5529,0.3355,0.3314,0.3311


In [14]:
beste_qual_run = qual_tabel.sort_values("F1 macro", ascending=False).iloc[0]["Run"]
# Zoekt de run met de hoogste F1 macro-score

beste_qual = qual_resultaten[beste_qual_run]
# Haalt de volledige resultaten van de beste quality-run op

logger.info(f"Beste quality-run op basis van F1 macro: {beste_qual_run}")
# Laat zien welke run het best presteerde

logger.info("Classification report van de beste quality-run:")
print(beste_qual["report"])
# Toont het classificatierapport met precision, recall en F1-score per klasse

qual_cm_df = pd.DataFrame(
    beste_qual["confusion_matrix"],
    index=[f"Werkelijk: {label}" for label in beste_qual["labels"]],
    columns=[f"Voorspeld: {label}" for label in beste_qual["labels"]]
)
# Zet de confusion matrix om in een overzichtelijke tabel

display(qual_cm_df)
# Toont de confusion matrix van de beste run

INFO - Beste quality-run op basis van F1 macro: Experiment 1
INFO - Classification report van de beste quality-run:


              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.25      0.33      0.29         3
           3       0.50      0.12      0.20         8
           4       0.38      0.33      0.36        45
           5       0.61      0.65      0.63       165
           6       0.49      0.52      0.51       146
           7       0.58      0.58      0.58       121
           8       0.60      0.64      0.62        70
           9       0.73      0.38      0.50        21
          10       0.20      0.17      0.18         6

    accuracy                           0.55       586
   macro avg       0.43      0.37      0.39       586
weighted avg       0.55      0.55      0.55       586



,Voorspeld: 1,Voorspeld: 2,Voorspeld: 3,Voorspeld: 4,Voorspeld: 5,Voorspeld: 6,Voorspeld: 7,Voorspeld: 8,Voorspeld: 9,Voorspeld: 10
Werkelijk: 1,0,1,0,0,0,0,0,0,0,0
Werkelijk: 2,0,1,0,2,0,0,0,0,0,0
Werkelijk: 3,0,1,1,5,1,0,0,0,0,0
Werkelijk: 4,0,1,1,15,21,6,1,0,0,0
Werkelijk: 5,0,0,0,13,107,40,5,0,0,0
Werkelijk: 6,0,0,0,4,42,76,23,1,0,0
Werkelijk: 7,0,0,0,0,4,30,70,16,1,0
Werkelijk: 8,0,0,0,0,0,2,21,45,1,1
Werkelijk: 9,0,0,0,0,0,0,0,10,8,3
Werkelijk: 10,0,0,0,0,0,0,1,3,1,1


In [15]:
logger.info(f"Datatypes van gekozen kolommen:\n{df[['SalePrice', 'Gr Liv Area', 'Neighborhood']].dtypes}")

INFO - Datatypes van gekozen kolommen:
SalePrice       int64
Gr Liv Area     int64
Neighborhood      str
dtype: object


### Korte conclusie model 2
- Dit classificatiemodel was complexer, omdat er 10 kwaliteitsniveaus voorspeld moesten worden.
- Hyperparameter tuning verbeterde de prestaties meer dan het direct toevoegen van extra features.
- Hoewel `Gr Liv Area` en `House Style` logisch leken, zorgden ze niet voor de beste macro F1-score.

Beste run voor `Overall Qual`: `Experiment 1`

## Stap 8 – Verantwoording van de experimenten

### Wat inspireerde de volgende experimenten?

#### Garage
1. Ik begon met de drie features waarvan ik verwachtte dat ze de meeste invloed zouden hebben.
2. Daarna heb ik eerst de hyperparameters aangepast door een diepere boom te testen.
3. Omdat dat nog niet voldoende verbetering gaf, heb ik `Year Built` toegevoegd als extra feature.
4. Toen dit betere resultaten opleverde, heb ik ook `min_samples_leaf` toegevoegd om het model verder te tunen.

#### Overall Qual
1. Ook hier begon ik met de drie features waarvan ik vooraf de meeste invloed verwachtte.
2. Omdat dit een moeilijker multi-class probleem is, heb ik eerst verschillende hyperparameters getest.
3. Daarna heb ik extra features toegevoegd, zoals `Gr Liv Area` en `House Style`.
4. Uit de resultaten bleek dat bij dit model vooral de hyperparameters de grootste verbetering opleverden.

### Eindconclusie
- Voor `Garage` was de beste run: `Experiment 3`
- Voor `Overall Qual` was de beste run: `Experiment 1`

De aangepaste configuraties zorgden niet altijd voor een hogere accuracy, maar wel voor een betere macro F1-score.
In deze situatie is dat belangrijker, omdat macro F1-score de prestaties over alle klassen eerlijker beoordeelt.